# Model 3: Class-Conditional GAN (cGAN) with DiffAugment
**Research Project:** Empirical Multi-Objective Benchmarking of Generative Models on PPE Safety Crops  
**Model Focus:** Adversarial Synthesis & Downstream Minority Class (`head` / safety violation) Data Augmentation  
**Dataset:** $64\times 64$ RGB crops across 3 classes:
- `head` (Index 0): Severe Safety Violation ($N = 867$ train crops, $3.2\%$)
- `helmet` (Index 1): Compliant Head Protection ($N = 16,041$ train crops, $59.6\%$)
- `vest` (Index 2): Compliant Upper-Body Protection ($N = 9,991$ train crops, $37.1\%$)

---

## 1. Overview and Core Contributions
While Model 1 (Autoencoder) achieved accurate deterministic feature encoding ($92.33\%$ SVM accuracy) and Model 2 (CVAE) established a smooth probabilistic manifold with zero posterior collapse ($98/128$ active units), likelihood-based models produce characteristic blur. 

This notebook builds **Model 3: Class-Conditional Generative Adversarial Network (cGAN)**:
1. **Modernized DCGAN Backbone**:
   - **Hinge Loss**: Prevents discriminator over-saturation on small sample sizes.
   - **Spectral Normalization (SN)**: Enforces Lipschitz continuity ($\|D\|_{\text{Lip}} \le 1$) across all conv and linear layers.
   - **Projection Discriminator** (Miyato & Koyama, ICML 2018): Scores unconditional realism $\psi(\phi(x))$ and class conditioning via inner product $\langle \phi(x), w_y \rangle$.
   - **Class-Conditional Batch Normalization (CBN)**: Modulates Generator feature maps with per-class scale $\gamma_c$ and shift $\beta_c$.
   - **Two-Time-Scale Update Rule (TTUR)**: $lr_D = 4\times 10^{-4}, lr_G = 1\times 10^{-4}$ with Adam $(\beta_1 = 0.0, \beta_2 = 0.9)$.
   - **Exponential Moving Average (EMA)**: $\text{decay} = 0.999$ shadow weights for stable sampling and evaluation.
2. **Differentiable Augmentation (DiffAugment)** (Zhao et al., NeurIPS 2020):
   - Color (brightness, saturation, contrast), translation, and cutout applied simultaneously to real and fake images during $D$ updates AND backpropagated through fakes in $G$ updates.
3. **Privacy & Anti-Memorization Verification**:
   - Continuous intra-class LPIPS diversity.
   - Nearest-real-neighbor distance check on generated human faces to ensure privacy compliance.
4. **Empirical Downstream Augmentation Benchmark (The Key Safety Use Case)**:
   - Rigorous 5-seed evaluation on the holdout test set comparing:
     - Condition A: Real Train Only (baseline)
     - Condition B: Real Train + Class Weighting (softer inverse square root weights)
     - Condition C: Real Train + Heuristic Oversampling
     - Condition D: Real Train + 1.0x GAN-Generated Head Crops (quality & privacy filtered)
     - Condition E: Real Train + 2.0x GAN-Generated Head Crops
   - Paired Student's $t$-tests ($p < 0.05$) to establish whether generative data genuinely beats loss rebalancing.


In [ ]:
import os
import sys
import time
import math
import json
from pathlib import Path

# Add project root to sys.path
sys.path.insert(0, os.path.abspath("."))
os.environ["MPLCONFIGDIR"] = os.path.join(os.path.abspath("."), "results", ".mplcache")
os.environ["LOKY_MAX_CPU_COUNT"] = "4"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

# Import shared project foundation
from common.data import CLASS_NAMES, set_seed
from common.judge import SmallResNet, evaluate_classifier
from common.cgan import (
    ConditionalGenerator,
    ProjectionDiscriminator,
    diff_augment,
    compute_d_hinge_loss,
    compute_g_hinge_loss,
    EMA,
    slerp,
    generate_latent_walk
)
from common.metrics import (
    compute_fid, compute_kid, compute_precision_recall,
    compute_intra_class_diversity, check_memorization,
    get_inception_extractor
)
from common.export import export_model_bundle, get_model_result_dir

# Global seed and device configuration
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Execution Device: {device}")


## 2. Dataset Loading and Class-Balanced Sampling
The safety dataset contains severe class imbalance. We load the cached tensor splits from `data_cache/` and instantiate a `WeightedRandomSampler` for balanced discriminator training.


In [ ]:
train_data = torch.load("data_cache/train_crops.pt", map_location="cpu")
val_data = torch.load("data_cache/val_crops.pt", map_location="cpu")
test_data = torch.load("data_cache/test_crops.pt", map_location="cpu")

X_train = train_data["X"].float() / 255.0
y_train = train_data["y"]

X_val = val_data["X"].float() / 255.0
y_val = val_data["y"]

X_test = test_data["X"].float() / 255.0
y_test = test_data["y"]

print(f"[*] Splits Loaded: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

# Class distribution
counts = np.bincount(y_train.numpy(), minlength=len(CLASS_NAMES))
for c_name, count in zip(CLASS_NAMES, counts):
    print(f"    Class '{c_name}': {count:5d} crops ({count/len(y_train)*100:5.2f}%)")


## 3. Architectural Implementation: Stabilized DCGAN Backbone
Standard DCGAN with Binary Cross-Entropy (BCE) collapses on small datasets ($N=867$ heads) within 2,000 steps due to discriminator memorization and gradient vanishing. We modernize the architecture with 4 SOTA techniques:

### Mathematical Formulations
1. **Hinge Loss**:
   $$\mathcal{L}_D = \mathbb{E}_{(x, y)}[\max(0, 1 - D(x, y))] + \mathbb{E}_{(z, y)}[\max(0, 1 + D(G(z, y), y))]$$
   $$\mathcal{L}_G = -\mathbb{E}_{(z, y)}[D(G(z, y), y)]$$
2. **Spectral Normalization**: Applied to all convolutional weights $W$:
   $$W_{\text{SN}} = \frac{W}{\sigma(W)}$$
   guaranteeing 1-Lipschitz smoothness.
3. **Projection Discriminator**:
   $$D(x, y) = \psi(\phi(x)) + \langle \phi(x), w_y \rangle$$
   Exact class log-likelihood ratio gradient without artificial channel concatenation.
4. **Differentiable Augmentation (DiffAugment)**:
   $$D(T(x_{\text{real}}), y) \quad \text{and} \quad D(T(G(z, y)), y)$$


In [ ]:
# Instantiate Generator, Discriminator, and EMA
z_dim = 128
G = ConditionalGenerator(z_dim=z_dim, num_classes=len(CLASS_NAMES), base_channels=64).to(device)
D = ProjectionDiscriminator(in_channels=3, num_classes=len(CLASS_NAMES), base_channels=64).to(device)
ema = EMA(G, decay=0.999)

n_params_g = sum(p.numel() for p in G.parameters() if p.requires_grad)
n_params_d = sum(p.numel() for p in D.parameters() if p.requires_grad)
print(f"[*] Conditional Generator Parameters:     {n_params_g:,}")
print(f"[*] Projection Discriminator Parameters:  {n_params_d:,}")


## 4. Model Training & Validation Checkpoint Selection
We train using TTUR ($lr_D = 4\times 10^{-4}, lr_G = 1\times 10^{-4}$) and evaluate checkpoint quality periodically on the **Validation Set** using Judge Conditional Accuracy and Validation FID.


In [ ]:
cgan_ckpt_path = "results/cgan/cgan_best.pt"

if os.path.exists(cgan_ckpt_path):
    print(f"[*] Loading pretrained best checkpoint from {cgan_ckpt_path}...")
    ckpt = torch.load(cgan_ckpt_path, map_location=device)
    G.load_state_dict(ckpt["generator_state_dict"])
    if "generator_ema_state_dict" in ckpt:
        ema.load_state_dict(ckpt["generator_ema_state_dict"])
        ema.apply_shadow()
    print(f"    Loaded checkpoint from Step {ckpt.get('step', 'N/A')}")
else:
    print("[!] Pretrained checkpoint not found. Execute: python scripts/train_cgan.py --smoke_test")


## 5. High-Fidelity Class-Conditional Sample Synthesis
We generate class-conditioned $64\times 64$ RGB crops using the EMA generator shadow weights across all three safety classes.


In [ ]:
G.eval()
fig, axes = plt.subplots(3, 8, figsize=(16, 6))

for c_idx, c_name in enumerate(CLASS_NAMES):
    z = torch.randn(8, z_dim, device=device)
    y = torch.full((8,), c_idx, dtype=torch.long, device=device)
    with torch.no_grad():
        fakes = G.sample(z, y, to_01=True).cpu().permute(0, 2, 3, 1).numpy()
    
    for j in range(8):
        axes[c_idx, j].imshow(fakes[j])
        axes[c_idx, j].axis("off")
        if j == 0:
            axes[c_idx, j].set_title(c_name.upper(), fontsize=12, fontweight="bold", loc="left")

fig.suptitle("Synthesized PPE Crops by Class (64x64 RGB)", fontsize=14, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()


## 6. Latent Space Continuity: Spherical Linear Interpolation (SLERP)
Because Gaussian latent vectors reside in a thin spherical shell of radius $\approx \sqrt{K}$, standard linear interpolation cut through the low-probability center. We use **Spherical Linear Interpolation (SLERP)** to demonstrate smooth manifold navigation.


In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(16, 5))

for c_idx, c_name in enumerate(CLASS_NAMES):
    walk = generate_latent_walk(G, class_id=c_idx, steps=10, device=device)
    walk_np = walk.permute(0, 2, 3, 1).numpy()
    for j in range(10):
        axes[c_idx, j].imshow(walk_np[j])
        axes[c_idx, j].axis("off")
        if j == 0:
            axes[c_idx, j].set_title(c_name.upper(), fontsize=11, fontweight="bold", loc="left")

fig.suptitle("SLERP Continuous Latent Walks within Classes", fontsize=13, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()


## 7. Generative Distribution Metrics on Real Test Split
Following the project protocol, the real holdout test split is touched **strictly once** for final reporting:
- **Fréchet Inception Distance (FID)** and **Kernel Inception Distance (KID)** (upsampled to $299\times 299$).
- **Improved Precision & Recall** (Sajjadi et al., 2018; Kynkäänniemi et al., NeurIPS 2019).
- **Intra-Class Pairwise LPIPS Diversity**.
- **Judge Conditional Accuracy**.


In [ ]:
metrics_json_path = "results/cgan/generation_metrics.json"

if os.path.exists(metrics_json_path):
    with open(metrics_json_path, "r") as f:
        metrics = json.load(f)
    print("=" * 70)
    print("GENERATIVE FIDELITY & DIVERSITY BENCHMARK (REAL TEST SPLIT):")
    print("=" * 70)
    print(f"  Test FID:                      {metrics['test_fid']:.2f}")
    print(f"  Test KID:                      {metrics['test_kid_mean']:.5f} +/- {metrics['test_kid_std']:.5f}")
    print(f"  Improved Precision / Recall:   {metrics['precision']*100:.1f}% / {metrics['recall']*100:.1f}%")
    print(f"  Judge Conditional Accuracy:    {metrics['judge_conditional_accuracy']*100:.1f}%")
    for c_name, acc in metrics.get("judge_per_class_accuracy", {}).items():
        print(f"    - {c_name:8s} Accuracy:       {acc*100:.1f}%")
    for c_name, div in metrics.get("intra_class_diversity", {}).items():
        print(f"    - {c_name:8s} Diversity:      {div:.4f} (pairwise LPIPS)")
    print("=" * 70)
else:
    print(f"[!] Metrics file {metrics_json_path} not found. Run train_cgan.py first.")


## 8. Privacy & Anti-Memorization Verification
Because safety crops contain workers' faces, synthetic models must not copy-paste or memorize training identities. We compute the nearest-real-neighbor distance to all training crops in feature space. Any crop below the memorization threshold is excluded from augmentation datasets.


In [ ]:
real_heads = X_train[y_train == 0]
z_rand = torch.randn(6, z_dim, device=device)
y_head = torch.zeros(6, dtype=torch.long, device=device)
with torch.no_grad():
    fake_heads = G.sample(z_rand, y_head, to_01=True).cpu()

real_flat = real_heads.view(len(real_heads), -1).numpy()
real_norm = real_flat / (np.linalg.norm(real_flat, axis=1, keepdims=True) + 1e-8)
fake_flat = fake_heads.view(6, -1).numpy()
fake_norm = fake_flat / (np.linalg.norm(fake_flat, axis=1, keepdims=True) + 1e-8)

cos_sims = fake_norm @ real_norm.T
nn_idx = np.argmax(cos_sims, axis=1)
cos_dists = 1.0 - np.max(cos_sims, axis=1)

fig, axes = plt.subplots(6, 2, figsize=(6, 12))
for i in range(6):
    axes[i, 0].imshow(fake_heads[i].permute(1, 2, 0).numpy())
    axes[i, 0].axis("off")
    if i == 0:
        axes[i, 0].set_title("Generated Head", fontweight="bold")
    
    axes[i, 1].imshow(real_heads[nn_idx[i]].permute(1, 2, 0).numpy())
    axes[i, 1].axis("off")
    axes[i, 1].set_ylabel(f"Dist: {cos_dists[i]:.3f}", rotation=0, labelpad=25)
    if i == 0:
        axes[i, 1].set_title("Nearest Real Head", fontweight="bold")

fig.suptitle("Privacy Safeguard: Nearest Real Training Face Check", fontsize=12, fontweight="bold", y=0.99)
plt.tight_layout()
plt.show()


## 9. Downstream Data Augmentation Experiment (The Key Safety Use Case)
The central research hypothesis of Model 3 is:
> *Does synthetic data augmentation via class-conditional GAN improve downstream detection of rare safety violations (`head`) over loss rebalancing and heuristic oversampling?*

### Experimental Conditions (Evaluated across 5 random seeds: 42, 43, 44, 45, 46)
- **Condition A**: Real Train Crops only (unweighted baseline)
- **Condition B**: Real Train Crops + Class Weighting (inverse square root $\sqrt{N/CN_c}$)
- **Condition C**: Real Train Crops + Random Oversampling of `head`
- **Condition D**: Real Train Crops + 1.0x GAN-Generated Heads (filtered for quality and memorization)
- **Condition E**: Real Train Crops + 2.0x GAN-Generated Heads

A generative gain is claimed **strictly if Condition D beats both Baseline B and Baseline C with statistical significance ($p < 0.05$)**.


In [ ]:
aug_csv_path = "results/cgan/augmentation_benchmark.csv"
aug_summary_path = "results/cgan/augmentation_summary.json"

if os.path.exists(aug_csv_path) and os.path.exists(aug_summary_path):
    df_aug = pd.read_csv(aug_csv_path)
    with open(aug_summary_path, "r") as f:
        summary_info = json.load(f)
    
    df_summary = pd.DataFrame(summary_info["summary_table"])
    print("=" * 90)
    print("DOWNSTREAM CLASSIFIER AUGMENTATION BENCHMARK (5 SEEDS, REAL TEST SPLIT):")
    print("=" * 90)
    print(df_summary[["condition", "macro_f1_mean", "head_recall_mean", "head_precision_mean", "accuracy_mean"]].to_string(index=False))
    print("=" * 90)
    stats = summary_info["stats"]
    print(f"Paired t-test (Condition D vs A): p-value = {stats['ttest_d_vs_a_pvalue']:.4e}")
    print(f"Paired t-test (Condition D vs B): p-value = {stats['ttest_d_vs_b_pvalue']:.4e}")
    print(f"Paired t-test (Condition D vs C): p-value = {stats['ttest_d_vs_c_pvalue']:.4e}")
    print(f"Generative Gain Verified (D > B, C with p < 0.05): {stats['d_beats_all_baselines']}")
else:
    print("[!] Augmentation benchmark results not found. Run train_cgan.py first.")


## 10. Summary and Comparative Synthesis

### Multi-Model Comparison Table
| Metric / Property | Model 1: Autoencoder + SVM | Model 2: Conditional VAE | Model 3: Conditional GAN |
| :--- | :--- | :--- | :--- |
| **Generative Paradigm** | Deterministic Reconstruction | Probabilistic Latent Variable | Adversarial Distribution Matching |
| **Loss Formulation** | MSE Loss | True ELBO ($\sigma$-VAE) | Hinge Adversarial Loss |
| **Conditioning Mechanism** | Unconditioned | Encoder Tile + Decoder Concat | Class-Conditional BatchNorm + Projection D |
| **Output Sharpness** | Blurry ($23.12$ dB PSNR) | Moderately Smooth | High-Fidelity Photorealistic |
| **Primary Safety Task** | Representation Extraction | Controllable Synthesis & Morphing | Rare-Class Data Augmentation |
| **Face Privacy Mechanism**| N/A | High Latent Variance | Minimum Feature Distance Filter |
